In [6]:
# Import Libraries and Setting up Header
import io
import time
import chess.pgn
import pandas as pd
import requests

headers = {
    'User-Agent': 'Cheese/1.0 (Contact: radityastudy@gmail.com)'
}

In [ ]:
# # Ambil semua pemain bergelar Grandmaster (GM)
# year = '2026'
# months = ['01', '02', '03', '04', '05', '06']

# print('Mengambil daftar Grandmaster...')
# gm_url = 'https://api.chess.com/pub/titled/GM'
# response = requests.get(gm_url, headers=headers)
# all_gms = response.json().get('players', [])

# target_players = all_gms
# print(f'Berhasil mendapatkan {len(all_gms)} GM.')

# valid_top_players = []
# MIN_GAMES_REQUIRED = 5  # Batas minimal game per bulan agar data di dashboard padat

# # Filter Pemain yang memiliki ELO >= 2800 DAN Aktif Bermain
# for username in target_players:
#     stats_url = f'https://api.chess.com/pub/player/{username}/stats'
#     res = requests.get(stats_url, headers=headers)
    
#     if res.status_code == 200:
#         stats = res.json()
#         # Ambil rating tertinggi dari Blitz atau Rapid
#         blitz_rating = stats.get('chess_blitz', {}).get('last', {}).get('rating', 0)
#         rapid_rating = stats.get('chess_rapid', {}).get('last', {}).get('rating', 0)
        
#         max_elo = max(blitz_rating, rapid_rating)
        
#         # 1. Cek Filter Pertama: Apakah ELO memenuhi syarat?
#         if max_elo >= 2800:
#             print(f'User: {username} LOLOS FILTER ELO ({max_elo}). Memeriksa keaktifan...')
            
#             # 2. Cek Filter Kedua: Pre-screening aktivitas game di bulan target
#             is_active_player = True
            
#             for month in months:
#                 check_url = f"https://api.chess.com/pub/player/{username}/games/{year}/{month}"
#                 games_res = requests.get(check_url, headers=headers)
                
#                 if games_res.status_code == 200:
#                     total_games = len(games_res.json().get('games', []))
                    
#                     # Jika ada satu bulan yang jumlah game-nya di bawah standar, batalkan player ini
#                     if total_games < MIN_GAMES_REQUIRED:
#                         is_active_player = False
#                         break  # Keluar dari loop bulan, lanjut ke GM berikutnya
#                 else:
#                     is_active_player = False
#                     break
                
#                 time.sleep(0.1)  # Jeda tipis saat cek bulan
            
#             # Jika lolos kedua filter, baru masukkan ke list valid
#             if is_active_player:
#                 valid_top_players.append(username)
                
#     # Delay dasar setelah memeriksa satu profil GM
#     time.sleep(0.2)

In [ ]:
# Ambil semua pemain bergelar Grandmaster (GM)
print('Mengambil daftar Grandmaster...')
gm_url = 'https://api.chess.com/pub/titled/GM'
response = requests.get(gm_url, headers=headers)
all_gms = response.json().get('players', [])

target_players = all_gms
print(f'Berhasil mendapatkan {len(all_gms)} GM.')

valid_top_players = []

# Filter Pemain yang memiliki ELO > 2800 di Blitz atau Rapid
for username in target_players:
    stats_url = f'https://api.chess.com/pub/player/{username}/stats'
    res = requests.get(stats_url, headers=headers)
    
    if res.status_code == 200:
        stats = res.json()
        # Ambil rating tertinggi dari Blitz atau Rapid
        blitz_rating = stats.get('chess_blitz', {}).get('last', {}).get('rating', 0)
        rapid_rating = stats.get('chess_rapid', {}).get('last', {}).get('rating', 0)
        
        max_elo = max(blitz_rating, rapid_rating)
        
        if max_elo >= 2800:
            print(f'User: {username} VALID (ELO: {max_elo})')
            valid_top_players.append(username)
    
    # Delay kecil agar tidak terkena Rate Limit (HTTP 429)
    time.sleep(0.2)

In [ ]:
# Tarik data game bulanan dari pemain yang lolos filter
all_games_list = []

for month in months:
    for player in valid_top_players:
        games_url = f"https://api.chess.com/pub/player/{player}/games/{year}/{month}"
        games_res = requests.get(games_url, headers=headers)
        
        if games_res.status_code == 200:
            games_data = games_res.json().get('games', [])
            print(f"Mengambil {len(games_data)} game dari {player} untuk bulan {month}-{year}")
            
            # Ekstrak komponen dari setiap game
            for game in games_data:
                eco_url = game.get('eco', '')

                # Parsing sederhana untuk membersihkan URL menjadi nama teks pembukaan
                if eco_url:
                    opening_name = eco_url.split('/openings/')[-1].replace('-', ' ')
                else:
                    opening_name = 'Unknown Opening'

                pgn_text = game.get('pgn', '')
                total_turns = 0
                
                if pgn_text:
                    pgn_io = io.StringIO(pgn_text)
                    parsed_game = chess.pgn.read_game(pgn_io)
                    
                    if parsed_game:
                        # Menghitung total ply (langkah tunggal putih atau hitam)
                        total_ply = sum(1 for _ in parsed_game.mainline())
                        # Konversi ply menjadi turn penuh standar catur
                        total_turns = (total_ply + 1) // 2

                game_features = {
                    'game_id': game.get('uuid'),
                    'time_class': game.get('time_class'),
                    'opening_name': opening_name,
                    'total_turns': total_turns,
                    'white_username': game.get('white', {}).get('username'),
                    'white_rating': game.get('white', {}).get('rating'),
                    'white_result': game.get('white', {}).get('result'),
                    'black_username': game.get('black', {}).get('username'),
                    'black_rating': game.get('black', {}).get('rating'),
                    'black_result': game.get('black', {}).get('result')
                }
                
                all_games_list.append(game_features)
                
        time.sleep(0.2)

df_games = pd.DataFrame(all_games_list)
df_games.drop_duplicates(subset=['game_id'], inplace=True)

df_games.to_csv('top_players_games.csv', index=False)
print(f"Total baris data yang siap diolah: {df_games.shape[0]}")

In [8]:
df_games.head()

,game_id,time_class,opening_name,total_turns,white_username,white_rating,white_result,black_username,black_rating,black_result
0,8efbd65f-e7f8-11f0-8ffe-11616601000f,blitz,Benoni Defense Modern Classical New York Varia...,55,0blivi0usspy,2921,win,Immortal_legend710,2830,timeout
1,a1d6f606-e812-11f0-9cfc-7c4ed201000f,blitz,Queens Gambit Declined Harrwitz Two Knights De...,70,TheFinal_Move,2944,win,0blivi0usspy,2913,timeout
2,a8a6abea-e823-11f0-8085-d502c301000f,blitz,Owens Defense...4.Qe2 e6 5.Nf3 d5 6.e5,29,0blivi0usspy,2919,win,bach12345_lfay,2825,resigned
3,4bdff3c4-e824-11f0-8ffe-11616601000f,blitz,Vienna Game Max Lange Vienna Gambit 3...exf4,39,bach12345_lfay,2827,repetition,0blivi0usspy,2917,repetition
4,8c42d8be-e889-11f0-9c69-fb411201000f,blitz,Queens Gambit Declined Exchange Positional Lin...,34,0blivi0usspy,2924,win,Thecarefreeangel,2876,resigned
